# IEEE118 EMT — deep solver and component profile

**Settings:** Δt = 50 µs, simulated duration = 10 s (200,000 steps) for detailed profiles; five clean 1-second runs for wall-time comparisons. All profiles use the same IEEE118 topology and conventional VBR generators. Data logging is disabled during timing.

The solver phase rows are nested inside the `Solve` task and must not be added to scheduler task totals. The dense/compact RHS callback counts below are component-stamp invocations summed over the whole run.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("data/IEEE118_DeepSolverProfile")
if not DATA.exists():
    DATA = Path("examples/Notebooks/Performance/data/IEEE118_DeepSolverProfile")
CASE_DIRS = {
    "Conventional": DATA / "conventional",
    "SSN": DATA / "ssn",
    "Aggregate": DATA / "aggregate",
    "Grouped shared-map": DATA / "grouped",
}


def load_case(folder):
    return {
        "tasks": pd.read_csv(next(folder.glob("*TaskTimes.csv"))),
        "steps": pd.read_csv(next(folder.glob("*TaskTimes_SchedulerSteps.csv"))),
        "phases": pd.read_csv(next(folder.glob("*SolverPhaseTimes.csv"))),
        "matrix": pd.read_csv(next(folder.glob("*SolverMatrixStats.csv"))),
    }


CASES = {name: load_case(folder) for name, folder in CASE_DIRS.items()}
CASES["Grouped shared-map"]["diagnostics"] = pd.read_csv(
    next(CASE_DIRS["Grouped shared-map"].glob("*SolverDiagnostics.csv"))
)
print("Loaded detailed 10-second profiles from:", DATA)

## Solver phases and sparse-matrix sizes

`partial_refactorization` is the solver's update phase. For the aggregate case the configured KLU path performs a full numeric refactor (partial refactorization is disabled); the profile label remains shared by the adapter interface.

In [ ]:
phase_names = [
    "rhs_assembly",
    "matrix_reset_and_variable_stamping",
    "partial_refactorization",
    "numeric_solve",
    "node_voltage_propagation",
]
phase_rows = []
matrix_rows = []
for case, data in CASES.items():
    phases = data["phases"].set_index("phase")
    for phase in phase_names:
        row = phases.loc[phase]
        phase_rows.append(
            {
                "case": case,
                "phase": phase,
                "total_s": row.total_s,
                "avg_us": row.avg_us,
                "calls": int(row.calls),
            }
        )
    matrix_rows.append({"case": case, **data["matrix"].iloc[0].to_dict()})
phase_summary = pd.DataFrame(phase_rows)
display(
    phase_summary.pivot(index="phase", columns="case", values="total_s").reindex(
        phase_names
    )
)
display(pd.DataFrame(matrix_rows).set_index("case"))

fig, ax = plt.subplots(figsize=(12, 5))
phase_summary.pivot(index="phase", columns="case", values="total_s").reindex(
    phase_names
).plot(kind="bar", ax=ax)
ax.set_ylabel("Cumulative time over 10 s simulation (s)")
ax.set_xlabel("Solver phase (nested inside Solve)")
ax.set_title("IEEE118 EMT solver-phase totals — Δt = 50 µs, T = 10 s")
ax.grid(axis="y", alpha=0.25)
ax.legend(title="Configuration")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()

## RHS assembly and function-call counts

The optimized reduced-order VBR generators now add only their three active RHS rows. This preserves the component ordering and arithmetic at each row while avoiding a full network-vector addition per generator. The paired before/after trace check was bit-for-bit identical across 10,001 samples and 28 logged channels.

In [ ]:
rhs_rows = []
for case, data in CASES.items():
    phases = data["phases"].set_index("phase")
    for phase in [
        "rhs_assembly",
        "rhs_dense_stamp_accumulation",
        "rhs_compact_stamp_callbacks",
    ]:
        rhs_rows.append(
            {"case": case, "phase": phase, "seconds": phases.loc[phase, "total_s"]}
        )
    for phase in [
        "rhs_dense_stamp_component_calls",
        "rhs_compact_stamp_component_calls",
    ]:
        rhs_rows.append(
            {"case": case, "phase": phase, "seconds": int(phases.loc[phase, "calls"])}
        )
rhs = pd.DataFrame(rhs_rows)
display(rhs.pivot(index="phase", columns="case", values="seconds"))

rhs_time = rhs[
    rhs.phase.isin(["rhs_dense_stamp_accumulation", "rhs_compact_stamp_callbacks"])
]
ax = rhs_time.pivot(index="case", columns="phase", values="seconds").plot(
    kind="bar", figsize=(10, 4)
)
ax.set_ylabel("Cumulative RHS accumulation time (s)")
ax.set_title("RHS assembly split by dense accumulation and compact callbacks")
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=15, ha="right")
plt.tight_layout()

## Component-type pre-step, post-step, and call volume

Per-task totals below are scheduler task time. A task row's call count is 200,000 for each component in this 10-second profile. Aggregated and grouped models intentionally replace hundreds of line tasks with one linear-network task.

In [ ]:
def component_kind(task):
    prefix = task.split(".", 1)[0]
    low = prefix.lower()
    if prefix.startswith("Gen"):
        return "Generators"
    if prefix.startswith("Line"):
        return "Lines"
    if prefix.startswith("Trafo"):
        return "Transformers"
    if "load" in low or "shunt" in low:
        return "Loads / shunts"
    if "linear" in low or "network" in low:
        return "Linear network block"
    return "Other"


component_rows = []
for case, data in CASES.items():
    tasks = data["tasks"].copy()
    tasks["kind"] = tasks.task.map(component_kind)
    tasks = tasks[tasks.task.str.endswith((".MnaPreStep", ".MnaPostStep"))].copy()
    tasks["stage"] = (
        tasks.task.str.rsplit(".", n=1)
        .str[-1]
        .map({"MnaPreStep": "Pre-step", "MnaPostStep": "Post-step"})
    )
    grouped = tasks.groupby(["kind", "stage"], as_index=False).agg(
        total_s=("total_ns", lambda x: x.sum() / 1e9),
        calls=("calls", "sum"),
        task_rows=("task", "count"),
    )
    grouped["case"] = case
    component_rows.append(grouped)
component_times = pd.concat(component_rows, ignore_index=True)
display(
    component_times.pivot_table(
        index=["kind", "stage"], columns="case", values="total_s", fill_value=0
    )
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, stage in zip(axes, ["Pre-step", "Post-step"]):
    subset = component_times[component_times.stage == stage]
    subset.pivot(index="kind", columns="case", values="total_s").plot(kind="bar", ax=ax)
    ax.set_title(stage)
    ax.set_xlabel("Component type / model block")
    ax.grid(axis="y", alpha=0.25)
    ax.tick_params(axis="x", rotation=25)
axes[0].set_ylabel("Cumulative task time over 10 s (s)")
axes[1].legend(title="Configuration", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.suptitle("Component pre/post task time by type — Δt = 50 µs, T = 10 s")
plt.tight_layout()

In [ ]:
# Largest individual scheduler tasks per configuration, including invocation counts.
top = []
for case, data in CASES.items():
    t = data["tasks"].copy()
    t["total_s"] = t.total_ns / 1e9
    t = t.sort_values("total_s", ascending=False).head(10)
    t.insert(0, "case", case)
    top.append(t[["case", "task", "calls", "avg_ns", "total_s"]])
top_tasks = pd.concat(top, ignore_index=True)
display(
    top_tasks.sort_values(["case", "total_s"], ascending=[True, False]).reset_index(
        drop=True
    )
)

call_summary = component_times.groupby(["case", "kind", "stage"], as_index=False).agg(
    calls=("calls", "sum"), task_rows=("task_rows", "sum"), total_s=("total_s", "sum")
)
display(call_summary.sort_values(["case", "total_s"], ascending=[True, False]))

## Grouped DenseLU details and solver selection

The grouped matrix is 162×162 and structurally dense (26,244 nonzeros). It has 486 declared variable generator entries. DenseLU is therefore the appropriate default: SparseLU and KLU partial refactorization were measured on this same grouped configuration.

In [ ]:
display(CASES["Grouped shared-map"]["diagnostics"])
runs = pd.read_csv(DATA / "five_run_timings.csv")
solver_summary = (
    runs.groupby(["configuration", "solver"])
    .time_s.agg(["mean", "std", "median", "min", "max"])
    .sort_values("mean")
)
display(solver_summary)

fig, ax = plt.subplots(figsize=(10, 4))
solver_summary["mean"].plot(
    kind="bar",
    yerr=solver_summary["std"],
    capsize=4,
    ax=ax,
    color=["#4c9f70", "#4285d4", "#e8a33a", "#c75b5b", "#8c6bb1", "#d4a6c8"],
)
ax.set_ylabel("Wall time for 1-second simulation (s)")
ax.set_title("Five-run solver timing — grouped shared-map case, Δt = 50 µs")
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()

## Findings

- Conventional PiLine child tasks dominate post-step work: 26.4 s summed over 10 s, versus 3.0 s for all-SSN lines.
- The fused aggregate has the lowest RHS cost (0.95 s) because it reduces 353 compact component callbacks per step to 55; conventional and all-SSN still pay for their separate line/load/transformer callbacks.
- Matrix updates/refactorization dominate conventional and ordinary SSN/aggregate solve work. The grouped matrix changes the balance: DenseLU numeric solve is 6.35 s, while its update phase is 1.70 s.
- Grouped post-step is not scheduler overhead: the single linear-network post task is 4.37 s and the profiled state-update multiply is 4.42 s.
- Grouped DenseLU used 398,082 correction solves (1.99 per MNA step) and one fallback refactorization over 200,000 steps. This refinement work explains why its solve bar remains appreciable.
- Five-run grouped solver means on the final build: DenseLU 2.21 s, SparseLU 7.52 s, KLU partial 17.33 s per simulated second. Sparsity is lost by the group reduction, so SparseLU/KLU are poor fits for this particular dense block.

The attempted variable-entry-only sparse matrix reset/KLU sync was not retained: a five-run A/B did not establish a speedup. The compact generator RHS path is retained because it eliminates full-size per-generator vector additions, reduces measured RHS time, and preserves bit-for-bit traces.